# Step 0: The big picture

> **How to use this notebook**
> - **Just reading?** The text cells contain the whole lesson. You can skip every code cell.
> - **Want to practice?** Run the code cells from top to bottom (Shift+Enter). They recompute every table and let you experiment. In **Your turn** cells, replace `None` with your answer: `check(...)` says if it is right and gives a hint if not, and `solution("0.1")` shows the worked answer.
> - Every question also has a **Show the answer** box you can open.

We build a **toddler LLM**: a model whose whole world is a few baby words. It is small enough that every number can be computed by hand, and every idea is shown on it.

In [ ]:
# Run this first (practice only): it makes the `toddler` package (repo root) importable
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

from toddler.data import CORPUS
from toddler.exercises import check, solution

## 0.1 The only thing an LLM does

**It reads some tokens and guesses the next one.** That's all it does.
Chatting, coding and answering questions are all this one guess, repeated in a loop.

> Mommy: "I am hungry. I want…"
> Toddler: "milk!"

The toddler doesn't follow a rule. It has heard these words many times, so after "hungry", "milk" is the most likely next word. An LLM works the same way.

## 0.2 The toddler's whole world

Its vocabulary has 10 words:

`.` · `I` · `am` · `want` · `hungry` · `sad` · `tired` · `milk` · `mommy` · `bed`

Everything it has ever heard (the training data, 3 sentences = 24 words):

```
I am hungry . I want milk .
I am sad . I want mommy .
I am tired . I want bed .
```

These sentences were picked on purpose. Each one hides an important idea:

| Situation | Why it's tricky | Idea it needs |
|---|---|---|
| After "I am": hungry? sad? tired? | Even a perfect model can't know | The model gives **probabilities** (about ⅓ each) |
| After "I want": milk? mommy? bed? | The answer depends on a word **3 steps back** | **Attention** |
| "I" is followed by "am" the 1st time and "want" the 2nd | Same word, different next word | **Context / position** |

In [ ]:
# Try it: the training data
for sentence in CORPUS:
    print(sentence)

words = [word for sentence in CORPUS for word in sentence.split()]
print(f"\n{len(words)} words heard, {len(set(words))} different: {sorted(set(words))}")

## 0.3 One guess, start to finish

```text
  "I am hungry . I want"
                │
                ▼
  ┌───────────────────────────┐
  │ 1. Tokenizer              │   text → pieces → IDs: 21 24 36 22 21 28
  └─────────────┬─────────────┘
                ▼
  ┌───────────────────────────┐
  │ 2. Embedding              │   each ID → a vector (a list of 8 numbers)
  └─────────────┬─────────────┘
                ▼
  ┌───────────────────────────┐
  │ 3. + Position             │   "I am token #1, #2, #3, ..."
  └─────────────┬─────────────┘
                ▼
  ┌───────────────────────────┐
  │ 4. Transformer blocks × N │   attention: each token looks at earlier tokens
  │                           │   feed-forward: each token thinks on its own
  └─────────────┬─────────────┘
                ▼
  ┌───────────────────────────┐
  │ 5. Output layer           │   one score for each token of the vocabulary
  └─────────────┬─────────────┘
                ▼
  ┌───────────────────────────┐
  │ 6. Softmax                │   scores → probabilities: milk 95%, mommy 3%, bed 2%, ...
  └─────────────┬─────────────┘
                ▼
  ┌───────────────────────────┐
  │ 7. Pick                   │   → "milk"
  └─────────────┬─────────────┘
                │
                └──► add "milk" to the text, then run everything again
```

1. **Tokenizer** (Step 1): cuts the text into pieces it knows and replaces each piece with its ID. The model never sees letters, only numbers.
2. **Embedding** (Step 2): turns each ID into a short list of numbers called a *vector*. Think of it as a spot on a "map of meaning". hungry, sad and tired will end up close to each other.
3. **Position** (Step 3): tells each vector "I'm token #1, #2, …". Without it, attention sees only a bag of words, so "I want milk" and "milk want I" look the same.
4. **Transformer blocks** (Steps 4 and 5): the brain. Each block does two things, and blocks are stacked:
   - **Attention:** each token looks back at earlier tokens and takes in what it needs ("want" takes in "hungry").
   - **Feed-forward:** each token "thinks" on its own about what it just took in.
5. **Output layer** (Step 6): turns the last token's vector into one score for each token of the vocabulary.
6. **Softmax** (Step 6): turns the scores into probabilities that add up to 100%.
7. **Pick** (Step 8): chooses a token (usually a likely one), adds it to the text, and runs everything again.

## 0.4 How the toddler learns (training)

At "birth" all its numbers are random, so it babbles: `bed am . mommy I`.
Training is a guessing game played on its own sentences. For the first sentence:

| It sees | Must guess |
|---|---|
| I | am |
| I am | hungry |
| I am hungry | . |
| I am hungry . | I |
| I am hungry . I | want |
| I am hungry . I want | milk |
| I am hungry . I want milk | . |

For each guess:
1. Measure how *surprised* it was by the right answer. This is the **loss**.
2. Change every number a tiny bit so the right answer becomes a little more likely. This is **gradient descent**. **Backpropagation** works out which way to change each number.
3. Repeat thousands of times.

Nobody ever tells it "hungry goes with milk." It is only rewarded for guessing well, and the only way to guess "milk" is to find that link itself. All of an LLM's "knowledge" comes from this.

In [ ]:
# Try it: every guess of the game, for the 3 sentences (7 guesses each)
games = []
for sentence in CORPUS:
    sentence_words = sentence.split()
    for i in range(1, len(sentence_words)):
        games.append((" ".join(sentence_words[:i]), sentence_words[i]))

for seen, target in games:
    print(f"{seen:<28} → must guess: {target}")
print(f"\n{len(games)} guesses in total")

## 0.5 Where is the knowledge kept?

Only in the numbers, called **parameters** or **weights**: the embedding table, the attention and feed-forward tables, and the output layer. No sentences are saved and no rules are written.

| | Toddler model | GPT-class model |
|---|---|---|
| Vocabulary | 53 tokens (10 used by our sentences) | ~32k–200k tokens |
| Vector size | 8 | thousands (4k–16k) |
| Transformer blocks | 1 | dozens to 100+ |
| Training data | 24 tokens | trillions of tokens |
| Parameters | about 2,000 | billions to hundreds of billions |

It's the same machine, just much bigger.

## 0.6 A first working model: just counting

Before building the real machine, we build the simplest possible next-word guesser. It looks **only at the last word** and counts what came after it in the training data. This is called a *bigram* model, and its "weights" are just these counts:

| After the word | the toddler heard next |
|---|---|
| `I` | `am` ×3, `want` ×3 |
| `am` | `hungry` ×1, `sad` ×1, `tired` ×1 |
| `hungry`, `sad`, `tired` | `.` ×1 each |
| `.` | `I` ×3 |
| `want` | `milk` ×1, `mommy` ×1, `bed` ×1 |
| `milk`, `mommy`, `bed` | `.` ×1 each |

To turn counts into **probabilities**, divide each count by the total for that word. Example: after `am` the toddler heard `hungry` 1 time out of 3, so P(hungry | am) = 1/3.

In [ ]:
# Try it: count what comes after each word
from collections import Counter, defaultdict

def count_next_words(corpus):
    counts = defaultdict(Counter)
    for sentence in corpus:
        sentence_words = sentence.split()
        for current, following in zip(sentence_words, sentence_words[1:]):
            counts[current][following] += 1
    return counts

next_counts = count_next_words(CORPUS)
for word, counter in next_counts.items():
    print(f"after {word!r:<9} heard: {dict(counter)}")

### Your turn 0.1
Using the counts above, what are the probabilities of the next word after `I`? Write numbers like `0.25`, not `25%`.

<details>
<summary>Show the answer</summary>

After `I`, the toddler heard `am` 3 times and `want` 3 times, 6 in total: P(am) = 3/6 = **0.5** and P(want) = 3/6 = **0.5**.
The counting model only sees `I`, so it can't tell the first `I` from the second.

</details>

In [ ]:
my_answer = {"am": None, "want": None}
check("0.1", my_answer)

All the probabilities of the counting model:

| After the word | Probabilities of the next word |
|---|---|
| `I` | am 50%, want 50% |
| `am` | hungry 33%, sad 33%, tired 33% |
| `hungry`, `sad`, `tired` | . 100% |
| `.` | I 100% |
| `want` | milk 33%, mommy 33%, bed 33% |
| `milk`, `mommy`, `bed` | . 100% |

In [ ]:
# Try it: turn counts into probabilities
def next_word_probabilities(counts, word):
    total = sum(counts[word].values())
    return {following: count / total for following, count in counts[word].items()}

for word in next_counts:
    probabilities = next_word_probabilities(next_counts, word)
    print(f"after {word!r:<9} → " + ", ".join(f"{w} {p:.0%}" for w, p in probabilities.items()))

## 0.7 Generating: guess, pick, append, repeat

This loop is exactly how ChatGPT writes, one token at a time. We **pick at random, following the probabilities**: a 50% word gets picked about half of the time. One run, step by step:

| Text so far | Last word | Probabilities of the next word | Picked |
|---|---|---|---|
| I | `I` | am 50%, want 50% | am |
| I am | `am` | hungry 33%, sad 33%, tired 33% | tired |
| I am tired | `tired` | . 100% | . |
| I am tired . | `.` | I 100% | I |
| I am tired . I | `I` | am 50%, want 50% | **am** |
| I am tired . I am | `am` | hungry 33%, sad 33%, tired 33% | sad |
| I am tired . I am sad | `sad` | . 100% | . |

Result: **"I am tired . I am sad ."** The second `I` should be followed by `want`, but the model only looked at `I`.

Eight more runs:

```
I want bed . I want mommy .
I am tired . I am sad .
I want bed . I want bed .
I am sad . I want milk .
I am hungry . I am sad .
I want bed . I want bed .
I want bed . I am sad .
I am hungry . I want mommy .
```

In [ ]:
# Try it: generate one sentence, showing every step
import random

def generate(counts, start="I", length=8, seed=None, verbose=False):
    rng = random.Random(seed)
    words = [start]
    while len(words) < length:
        probabilities = next_word_probabilities(counts, words[-1])
        picked = rng.choices(list(probabilities), weights=list(probabilities.values()))[0]
        if verbose:
            options = ", ".join(f"{w} {p:.0%}" for w, p in probabilities.items())
            print(f"{' '.join(words):<26} | last word {words[-1]!r:<8} → {options:<30} → picked {picked!r}")
        words.append(picked)
    return " ".join(words)

print("\nresult:", generate(next_counts, seed=1, verbose=True))

In [ ]:
# Try it: eight more runs (change the seeds to get other sentences)
for seed in range(8):
    print(generate(next_counts, seed=seed))

## 0.8 Why counting is not enough

Look at the sentences above. There are two problems:

1. After `want`, the model only sees `want`, so it gives milk, mommy and bed ⅓ each. It **cannot look back** at `hungry`. The fix is **attention** (Step 4).
2. After `I`, it says am 50% / want 50%. It **cannot tell** the first `I` from the second. The fix is **context and positions** (Steps 3 and 4).

### Your turn 0.2
After `I am hungry . I want`, what probability does the counting model give to `milk`? And what would a perfect model give?

<details>
<summary>Show the answer</summary>

- Counting model: **1/3**. It only sees `want`, and it heard want → milk, want → mommy and want → bed once each.
- Perfect model: **1.0**. It also sees `hungry`, so the answer can only be milk.

</details>

In [ ]:
my_answer = {"counting model": None, "perfect model": None}
check("0.2", my_answer)

### Your turn 0.3 (challenge)
What fraction of generated 8-word sentences are exactly one of the 3 training sentences? Work it out with probabilities.

<details>
<summary>Show the answer</summary>

Go along the sentence and multiply the probability of each guess being right:

| Guess | Probability it is right |
|---|---|
| `I` → `am` | 1/2 |
| `am` → any feeling | 1 (any feeling is fine, as long as the thing matches it later) |
| feeling → `.` | 1 |
| `.` → `I` | 1 |
| `I` → `want` | 1/2 |
| `want` → the thing that matches the feeling | 1/3 |
| thing → `.` | 1 |

Total: 1/2 × 1/2 × 1/3 = **1/12 ≈ 0.083**. Generating 10,000 sentences gives about 8%, as predicted.

</details>

In [ ]:
my_answer = None  # a number between 0 and 1
check("0.3", my_answer)

In [ ]:
# Try it: measure it on 10,000 generated sentences
training_sentences = set(CORPUS)
samples = [generate(next_counts, seed=seed) for seed in range(10_000)]
share = sum(sample in training_sentences for sample in samples) / len(samples)
print(f"{share:.1%} of 10,000 generated sentences are training sentences")

### Experiment
The model's probabilities simply mirror how often it heard things. For example, if the toddler hears "I am hungry . I want milk ." one more time:

| After the word | Before | After hearing the sentence again |
|---|---|---|
| `am` | hungry 33%, sad 33%, tired 33% | hungry **50%**, sad 25%, tired 25% |
| `want` | milk 33%, mommy 33%, bed 33% | milk **50%**, mommy 25%, bed 25% |

Practice: add your own sentences in the cell below (you can even invent new words), re-run it, and watch the probabilities change.

In [ ]:
# Try it: change the training data
my_corpus = CORPUS + [
    "I am hungry . I want milk .",
]
my_counts = count_next_words(my_corpus)
for word in ["I", "am", "want"]:
    probabilities = next_word_probabilities(my_counts, word)
    print(f"after {word!r:<6} → " + ", ".join(f"{w} {p:.0%}" for w, p in probabilities.items()))
print()
for seed in range(5):
    print(generate(my_counts, seed=seed))

## Check your understanding

**1. What is the *only* thing the model does?**

<details>
<summary>Show the answer</summary>

It guesses the next token. Everything else (chatting, writing code, answering) is that one guess, repeated in a loop.

</details>

**2. After "I am", why can't even a perfect model be sure? What does it give instead?**

<details>
<summary>Show the answer</summary>

The text has no clue yet: hungry, sad and tired each came after "I am" once. So the best honest answer is a set of probabilities: about ⅓ each. A model that said "hungry 100%" would be wrong 2 times out of 3.

</details>

**3. Where is the toddler's knowledge kept after training?**

<details>
<summary>Show the answer</summary>

Only in its numbers: the **weights** (parameters). No sentence and no rule is stored anywhere. In the counting model of this notebook, the weights are simply the counts table.

</details>

## Summary

- An LLM only **guesses the next token**. Writing is guess → pick → append → repeat.
- It gives **probabilities**, not one answer. When the text gives no clue (after `am`), the best answer is to share the probability.
- Its knowledge lives in its **weights**. In the counting model, the weights are the counts table.
- Counting fails because it only looks at **the last word**. The transformer fixes this by looking back at the whole text.

## Roadmap

| Step | Topic | Question it answers |
|---|---|---|
| **0** | **Big picture** | (this notebook) |
| 1 | Tokenization | How does text become numbers? |
| 2 | Embeddings | How can a number carry meaning? |
| 3 | Positions | How does it know word order? |
| 4 | Attention (Q, K, V) | How does "want" find "hungry"? |
| 5 | Transformer block | Feed-forward, residuals, normalization, stacking |
| 6 | Output & softmax | How do vectors become "milk 95%"? |
| 7 | Training | How do random numbers become knowledge? |
| 8 | Generation | Sampling, temperature, the loop |
| 9 | Toddler → ChatGPT | Scale, fine-tuning, RLHF |

Next: **Step 1, tokenization** (`01_tokenization_bpe.ipynb`).